In [ ]:
# 0. Instalar dependência zstd primeiro
!apt-get install -y zstd

In [ ]:
# 1. Instalar o Ollama no Colab
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Iniciar o servidor em background
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

In [ ]:
# 3. Baixar o modelo Llama 3.1 1B
!ollama pull llama3.2:1b


In [ ]:
# 4. Usar o modelo
!pip install ollama -q

import ollama

resposta = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "system",
            "content": "Você é um sistema de controle de missão espacial. Analise os dados e indique alertas."
        },
        {
            "role": "user",
            "content": "Temperatura: 95°C | Energia: 18% | Comunicação: instável. Avalie o status."
        }
    ]
)

print(resposta["message"]["content"])

In [ ]:
import random
import datetime

def gerar_dados_missao():
    """
    Gera dados simulados aleatórios dos sensores da missão espacial.
    Retorna um dicionário com todos os parâmetros operacionais.
    """
    # Temperatura dos módulos (°C) — normal: 20-75°C, crítico: >85°C
    temperatura_modulo_principal = round(random.uniform(15, 110), 1)
    temperatura_modulo_propulsao = round(random.uniform(30, 120), 1)
    temperatura_modulo_energia   = round(random.uniform(10, 90), 1)

    # Energia — nível de bateria (%) e geração solar (W)
    nivel_bateria   = round(random.uniform(5, 100), 1)
    geracao_solar   = round(random.uniform(0, 500), 1)

    # Comunicação — qualidade do sinal (%)
    qualidade_sinal = round(random.uniform(0, 100), 1)
    if qualidade_sinal >= 70:
        status_comunicacao = "estável"
    elif qualidade_sinal >= 35:
        status_comunicacao = "instável"
    else:
        status_comunicacao = "crítico"

    # Status dos módulos operacionais
    opcoes_status = ["operacional", "operacional", "operacional", "degradado", "falha"]
    modulos = {
        "Propulsão"      : random.choice(opcoes_status),
        "Suporte de vida" : random.choice(opcoes_status),
        "Navegação"      : random.choice(opcoes_status),
        "Comunicação"    : random.choice(opcoes_status),
        "Energia"        : random.choice(opcoes_status),
    }

    return {
        "timestamp"                  : datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "temperatura_principal"      : temperatura_modulo_principal,
        "temperatura_propulsao"      : temperatura_modulo_propulsao,
        "temperatura_energia"        : temperatura_modulo_energia,
        "nivel_bateria"              : nivel_bateria,
        "geracao_solar"              : geracao_solar,
        "qualidade_sinal"            : qualidade_sinal,
        "status_comunicacao"         : status_comunicacao,
        "modulos"                    : modulos,
    }

print("✅ Gerador de dados carregado!")

In [ ]:
def verificar_alertas(dados):
    """
    Verifica os dados da missão e retorna uma lista de alertas críticos.
    Classifica cada alerta por nível: CRÍTICO, AVISO ou OK.
    """
    alertas = []

    # --- Temperatura ---
    if dados["temperatura_principal"] > 85:
        alertas.append(("CRÍTICO", f"🔴 Temperatura módulo principal: {dados['temperatura_principal']}°C (limite: 85°C)"))
    elif dados["temperatura_principal"] > 70:
        alertas.append(("AVISO",   f"🟡 Temperatura módulo principal elevada: {dados['temperatura_principal']}°C"))

    if dados["temperatura_propulsao"] > 95:
        alertas.append(("CRÍTICO", f"🔴 Temperatura propulsão: {dados['temperatura_propulsao']}°C (limite: 95°C)"))
    elif dados["temperatura_propulsao"] > 80:
        alertas.append(("AVISO",   f"🟡 Temperatura propulsão elevada: {dados['temperatura_propulsao']}°C"))

    # --- Energia ---
    if dados["nivel_bateria"] < 15:
        alertas.append(("CRÍTICO", f"🔴 Nível de bateria crítico: {dados['nivel_bateria']}%"))
    elif dados["nivel_bateria"] < 30:
        alertas.append(("AVISO",   f"🟡 Nível de bateria baixo: {dados['nivel_bateria']}%"))

    if dados["geracao_solar"] < 50:
        alertas.append(("AVISO",   f"🟡 Geração solar baixa: {dados['geracao_solar']}W"))

    # --- Comunicação ---
    if dados["status_comunicacao"] == "crítico":
        alertas.append(("CRÍTICO", f"🔴 Sinal de comunicação crítico: {dados['qualidade_sinal']}%"))
    elif dados["status_comunicacao"] == "instável":
        alertas.append(("AVISO",   f"🟡 Sinal de comunicação instável: {dados['qualidade_sinal']}%"))

    # --- Módulos ---
    for modulo, status in dados["modulos"].items():
        if status == "falha":
            alertas.append(("CRÍTICO", f"🔴 FALHA no módulo: {modulo}"))
        elif status == "degradado":
            alertas.append(("AVISO",   f"🟡 Módulo degradado: {modulo}"))

    if not alertas:
        alertas.append(("OK", "🟢 Todos os sistemas operando dentro dos parâmetros normais."))

    return alertas

print("✅ Sistema de alertas carregado!")

In [ ]:
import ollama

def analisar_com_ia(dados, alertas):
    """
    Envia os dados da missão para o modelo Llama e retorna
    uma análise em linguagem natural com recomendações.
    """
    # Montar resumo dos alertas para o prompt
    resumo_alertas = "\n".join([f"- [{nivel}] {msg}" for nivel, msg in alertas])

    # Montar resumo dos módulos
    resumo_modulos = "\n".join([f"- {m}: {s}" for m, s in dados['modulos'].items()])

    prompt = f"""Dados da missão espacial registrados em {dados['timestamp']}:

TEMPERATURAS:
- Módulo principal: {dados['temperatura_principal']}°C
- Módulo propulsão: {dados['temperatura_propulsao']}°C
- Módulo energia: {dados['temperatura_energia']}°C

ENERGIA:
- Bateria: {dados['nivel_bateria']}%
- Geração solar: {dados['geracao_solar']}W

COMUNICAÇÃO:
- Qualidade do sinal: {dados['qualidade_sinal']}%
- Status: {dados['status_comunicacao']}

STATUS DOS MÓDULOS:
{resumo_modulos}

ALERTAS DETECTADOS:
{resumo_alertas}

Com base nesses dados, forneça:
1. Diagnóstico geral da missão (1-2 frases)
2. Principais riscos identificados
3. Ações recomendadas para a equipe de controle
Seja direto e técnico. Responda em português."""

    resposta = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Você é ARIA (Autonomous Response Intelligence for Aerospace), "
                    "sistema de IA de controle de missão espacial. "
                    "Analise dados operacionais e forneça diagnósticos precisos e recomendações técnicas. "
                    "Seja objetivo e priorize a segurança da missão."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return resposta["message"]["content"]

print("✅ Módulo de IA carregado!")

In [ ]:
def exibir_dashboard(dados, alertas, analise_ia):
    """
    Exibe o painel completo de controle da missão no terminal.
    """
    largura = 55

    print("\n" + "═" * largura)
    print("  🚀  MISSION CONTROL AI  —  ARIA v1.0")
    print("═" * largura)
    print(f"  📅 Timestamp : {dados['timestamp']}")
    print("─" * largura)

    # Temperaturas
    print("  🌡️  TEMPERATURAS")
    print(f"     Módulo Principal  : {dados['temperatura_principal']}°C")
    print(f"     Módulo Propulsão  : {dados['temperatura_propulsao']}°C")
    print(f"     Módulo Energia    : {dados['temperatura_energia']}°C")
    print("─" * largura)

    # Energia
    print("  ⚡ ENERGIA")
    barra = int(dados['nivel_bateria'] / 5)
    barra_visual = "█" * barra + "░" * (20 - barra)
    print(f"     Bateria    : [{barra_visual}] {dados['nivel_bateria']}%")
    print(f"     Solar      : {dados['geracao_solar']}W")
    print("─" * largura)

    # Comunicação
    print("  📡 COMUNICAÇÃO")
    print(f"     Sinal      : {dados['qualidade_sinal']}%  ({dados['status_comunicacao'].upper()})")
    print("─" * largura)

    # Status dos módulos
    print("  🔧 STATUS DOS MÓDULOS")
    icones = {"operacional": "✅", "degradado": "⚠️ ", "falha": "❌"}
    for modulo, status in dados["modulos"].items():
        icone = icones.get(status, "❓")
        print(f"     {icone} {modulo:<18}: {status.upper()}")
    print("─" * largura)

    # Alertas
    print("  🚨 ALERTAS DO SISTEMA")
    tem_critico = any(nivel == "CRÍTICO" for nivel, _ in alertas)
    for nivel, msg in alertas:
        print(f"     {msg}")
    print("─" * largura)

    # Análise da IA
    print("  🤖 ANÁLISE DA IA — ARIA")
    # Quebrar texto longo em linhas de até 50 chars
    palavras = analise_ia.split()
    linha_atual = "     "
    for palavra in palavras:
        if len(linha_atual) + len(palavra) + 1 > largura:
            print(linha_atual)
            linha_atual = "     " + palavra
        else:
            linha_atual += (" " if linha_atual.strip() else "") + palavra
    if linha_atual.strip():
        print(linha_atual)

    print("═" * largura)

    # Status geral
    if tem_critico:
        print("  🔴 STATUS GERAL: SITUAÇÃO CRÍTICA — INTERVENÇÃO NECESSÁRIA")
    elif any(nivel == "AVISO" for nivel, _ in alertas):
        print("  🟡 STATUS GERAL: ATENÇÃO — MONITORAMENTO REFORÇADO")
    else:
        print("  🟢 STATUS GERAL: MISSÃO NOMINAL — TODOS OS SISTEMAS OK")
    print("═" * largura + "\n")

print("✅ Interface de exibição carregada!")

In [ ]:
print("🚀 Iniciando Mission Control AI...\n")

# 1. Gerar dados simulados da missão
print("📡 Coletando dados dos sensores...")
dados = gerar_dados_missao()

# 2. Verificar alertas automáticos
print("🚨 Verificando alertas...")
alertas = verificar_alertas(dados)

# 3. Analisar com a IA
print("🤖 Consultando ARIA (IA)... (pode levar alguns segundos)\n")
analise = analisar_com_ia(dados, alertas)

# 4. Exibir dashboard
exibir_dashboard(dados, alertas, analise)